## Transform Customer Data
### 1.Remove the Records with Null Customer_id
### 2.Reomove Extact duplicates Records
### 3.Remove the Duplicate based on created Timestamps
### 4.Cast the correct data Types
### 5.Transform to silver Schema


In [0]:
%python
dfCustomers=spark.table("gizmobox_sankara.bronze.py_customers")
display(dfCustomers)

In [0]:
%python
df_Distinct_Customers=dfCustomers.filter('customer_id is not null and customer_name is not null')
display(df_Distinct_Customers)

In [0]:
%python
df_Distinct_Customers=dfCustomers.filter(dfCustomers.customer_id.isNotNull())
display(df_Distinct_Customers)

In [0]:
SELECT * FROM gizmobox_sankara.bronze.v_customers
WHERE customer_id IS NOT NULL

**2.Remove Records Exact Duplicate Records**

In [0]:
%python
df_Distinct_CustomersData=df_Distinct_Customers.distinct()
display(df_Distinct_CustomersData)

In [0]:
%python
df_Distinct_CustomersData=df_Distinct_Customers.dropDuplicates()
display(df_Distinct_CustomersData)

**3.Remove Duplicate Records Based On Created Timestamp**

In [0]:
%python
from pyspark.sql.functions import max,min
dt_max_ts=df_Distinct_Customers.groupBy('customer_id').agg(max('created_timestamp').alias('max_created_timestamp'))
display(dt_max_ts)

In [0]:
CREATE OR REPLACE TEMPORARY VIEW v_customers_distinct AS
SELECT DISTINCT * 
FROM gizmobox_sankara.bronze.v_customers
WHERE customer_id IS NOT NULL
ORDER BY customer_id

In [0]:
SELECT customer_id,MAX(created_timestamp)
FROM v_customers_distinct
GROUP BY customer_id

In [0]:
%python

df_Distinct_Customers_TimeStampData=(df_Distinct_CustomersData
.join(dt_max_ts,(df_Distinct_CustomersData.customer_id==dt_max_ts.customer_id) & 
      (df_Distinct_CustomersData.created_timestamp==dt_max_ts.max_created_timestamp),"inner").select(['*'])
)
display(df_Distinct_Customers_TimeStampData)

In [0]:
%python
df_temp = df_Distinct_Customers_TimeStampData.drop(dt_max_ts["customer_id"])
df_customer_casting=df_temp.select(
    df_temp.customer_id.cast('long').alias('Id'),
    df_temp.created_timestamp.cast('date')
)
display(df_customer_casting)

In [0]:
with cte_distinct as
(
SELECT customer_id,MAX(created_timestamp) as max_created_timestamp
FROM v_customers_distinct
GROUP BY customer_id
)
SELECT *
FROM v_customers_distinct c
join cte_distinct t on c.customer_id = t.customer_id and c.created_timestamp = t.max_created_timestamp

**4.CAST The Columns Into Correct Data Types**

In [0]:
with cte_distinct as
(
SELECT customer_id,MAX(created_timestamp) as max_created_timestamp
FROM v_customers_distinct
GROUP BY customer_id
)
SELECT CAST(t.max_created_timestamp AS TIMESTAMP),
c.customer_id,
c.customer_name,
CAST(c.date_of_birth AS TIMESTAMP) date_of_birth,
c.email,
c.telephone,
c.member_since,
c.customer_name,
c.FilePath
FROM v_customers_distinct c
join cte_distinct t on c.customer_id = t.customer_id and c.created_timestamp = t.max_created_timestamp

**5.Write Transformed Data To Silver Schema**

In [0]:
%python
df_Distinct_Customers_TimeStampData.drop(dt_max_ts["customer_id"]).writeTo('gizmobox_sankara.silver.py_customers').createOrReplace()
display(spark.table('gizmobox_sankara.silver.py_customers'))


In [0]:
SELECT * FROM gizmobox_sankara.silver.py_customers